In [ ]:
# =====================================================================================
# ENMIENDA C. H2 exploratoria para el índice simple y para Isolation Forest, más las
#   aclaraciones de implementación de H2 y H3. Se corre UNA sola vez, ANTES del cuaderno 18:
#   antes de abrir el archivo del SRI y antes de sortear la muestra de H3.
#   Solo escribe un archivo de constancia. No lee ninguna marca del SRI.
# =====================================================================================
import os, json, glob, hashlib
from datetime import datetime

SERCOP = r"C:\SERCOP_FINAL"
ENMIENDAS = os.path.join(SERCOP, "enmiendas")
PREREG = os.path.join(SERCOP, "preregistro")
RUC_SIN_MARCA_VIGENTE = "RUC_OCULTO_14"     # el contribuyente del archivo sin marca vigente al 9-oct-2026

def sha(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for bloque in iter(lambda: f.read(1 << 20), b""):
            h.update(bloque)
    return h.hexdigest()

# 1) Tiene que existir la firma del preregistro y el resultado de H1
firma_path = os.path.join(PREREG, "preregistro_firma.json")
if not os.path.exists(firma_path):
    raise SystemExit("No encuentro preregistro_firma.json: la enmienda C va después de la firma.")
FIRMA = json.load(open(firma_path, encoding="utf-8"))
regs_h1 = sorted(glob.glob(os.path.join(SERCOP, "modelos", "*_h1final", "registro_h1.json")))
if not regs_h1:
    raise SystemExit("No encuentro registro_h1.json de la corrida de H1 (carpeta *_h1final).")
REG_H1 = regs_h1[-1]
H1 = json.load(open(REG_H1, encoding="utf-8"))["h1"]

# 2) Tiene que ser anterior al cuaderno 18 y no debe existir otra enmienda C
if glob.glob(os.path.join(SERCOP, "modelos", "*_h2_h3", "registro_h3_muestra.json")) or \
   glob.glob(os.path.join(SERCOP, "modelos", "*_h2_h3", "h2_*")):
    raise SystemExit("Ya existe una carpeta del cuaderno 18 (*_h2_h3): la enmienda C debía ir antes. Avísame.")
if glob.glob(os.path.join(ENMIENDAS, "enmienda_*_c.json")):
    raise SystemExit("Ya existe una enmienda C: no se sobrescribe. Avísame si hay que corregirla.")
enm_b = os.path.join(ENMIENDAS, "enmienda_2026-10-16_b.json")

ahora = datetime.now()
enm = {
    "fecha": ahora.isoformat(timespec="seconds"),
    "id": f"enmienda_{ahora:%Y-%m-%d}_c",
    "sigue_a": {"preregistro": FIRMA["huella_documento"], "firma_fecha": FIRMA["fecha_firma"],
                "enmienda_b": sha(enm_b), "registro_h1": sha(REG_H1),
                "corrida_h1": os.path.basename(os.path.dirname(REG_H1))},
    "declaracion": ("Al registrar esta enmienda, el archivo planA_validacion_sri no se ha cargado en ningún "
                    "cuaderno y la muestra de H3 no se ha sorteado."),
    "motivo": (f"H1 no se cumplió (D = {H1['D']:+.4f}, p = {H1['p_unilateral']:.4f}): por la regla del "
               "preregistro el resultado favorece al índice simple. Interesa saber si la validación externa "
               "apunta en la misma dirección y si la ventaja descriptiva de Isolation Forest en H1 se repite."),
    "cambio_H2_exploratoria": {
        "que": ("H2 se calcula también con el puntaje del índice simple (columna 'base') y con el de Isolation "
                "Forest (columna 'if') de la lista congelada."),
        "como": ("Mismo procedimiento que la H2 principal: percentil a mitad de salto dentro de grupo x año, "
                 "T = mediana, permutación por proveedor con 9.999 réplicas y los mismos proveedores sorteados "
                 "en cada réplica; valor crítico registrado antes de calcular T."),
        "estatus": ("Exploratorio. No entra en la corrección de Holm ni en la decisión general. Se informa con "
                    "su valor p sin corregir y con la etiqueta 'exploratorio'."),
        "no_cambia": ("H2 principal (ensamble, 14 contribuyentes, grupo x año), sus sensibilidades y H3 quedan "
                      "como en el preregistro."),
    },
    "aclaraciones_H2": {
        "contribuyentes": ("Los 14 con marca vigente al 9-oct-2026 son los 15 del archivo planA_validacion_sri "
                           "menos uno sin marca vigente (verificado en el SRI). Su RUC está en el código del "
                           "cuaderno 18; aquí se guarda solo su huella."),
        "huella_ruc_excluido": hashlib.sha256(RUC_SIN_MARCA_VIGENTE.encode()).hexdigest(),
        "poblacion": ("Procesos de la lista congelada del período anterior a la reforma (desde 2016) con "
                      "puntaje del ensamble. Los procesos de 2015 y los de grupos no modelados no tienen "
                      "puntaje y no entran."),
        "tramos": "Número de procesos de cada proveedor dentro de esa población: 1; 2-3; 4-7; 8-15; 16-31; 32 o más.",
        "donantes": ("Proveedores con procesos en la población que no están entre los 15 del archivo del SRI. "
                     "En cada réplica se sortean sin reposición, dentro de cada tramo, tantos donantes como "
                     "contribuyentes tenga ese tramo. Generador numpy default_rng(20261001)."),
        "sin_procesos": "Un contribuyente sin procesos en la población no aporta a T ni se reemplaza.",
        "valor_critico": ("Percentiles 95 y 97,5 de la distribución nula de cada variante, guardados con huella "
                          "antes de calcular cualquier T."),
        "decil_monto": ("Percentil a mitad de salto de monto_final dentro de grupo x año, cortado en 10 tramos "
                        "iguales (1 a 10)."),
        "sensibilidad_15": "Sorteo propio con los tramos de los 15 contribuyentes y la misma semilla.",
        "incumplidos": "El registro de contratistas incumplidos no se obtuvo: esa sensibilidad no se calcula.",
    },
    "aclaraciones_H3": {
        "elegibles": "Procesos de la lista congelada del período anterior a la reforma con año 2016 a 2024.",
        "priorizados": ("Se ordenan por ocid y se permutan con default_rng(20261001); se aceptan en ese orden si no "
                        "superan 2 por proveedor ni 2 por entidad, hasta 60."),
        "controles": ("Procesos con priorizado = falso, ordenados por ocid y permutados con default_rng(20261001). "
                      "Para cada priorizado, en el orden de aceptación, se toma el primero del mismo grupo, año y "
                      "decil de monto que no supere los topes del lado control. Si no hay, decil a distancia 1 "
                      "(cualquiera de los dos), luego 2, y se declara."),
        "topes": "2 por proveedor y 2 por entidad en cada lado, contados por separado. Las reservas cuentan para los topes.",
        "reservas": ("20 priorizados de reserva (los siguientes en el mismo orden) y 2 controles de reserva por "
                     "cada control (mismo grupo, año y decil). Se usan solo si un proceso no es verificable."),
        "ceguera": ("Los 120 procesos y las reservas reciben un código neutro sorteado (R-001, R-002...). Los "
                    "revisores reciben solo código, ocid, código del proceso, familia y año. La clave (lado de cada "
                    "código) queda en un archivo aparte con huella y no se comparte hasta terminar la revisión."),
        "doble_revision": ("24 de los 120 códigos, sorteados con default_rng(20261001), van a los dos revisores; los "
                           "otros 96 se reparten al azar, 48 y 48."),
        "quien_prepara": "Quien corre el cuaderno 18 prepara la muestra y no revisa procesos.",
    },
}
arch = os.path.join(ENMIENDAS, f"enmienda_{ahora:%Y-%m-%d}_c.json")
with open(arch, "w", encoding="utf-8") as f:
    json.dump(enm, f, ensure_ascii=False, indent=1)
print("ENMIENDA C GUARDADA")
print("  archivo:", arch)
print("  huella: ", sha(arch))
print("  fecha:  ", enm["fecha"])
print("Mándate esta huella por correo (o al grupo) antes de correr el cuaderno 18: la fecha del correo")
print("prueba que la enmienda existía antes de abrir el archivo del SRI.")

In [ ]:
# =====================================================================================
# CELDA 1. Configuración y verificaciones (cuaderno 18: muestra de H3 y prueba de H2)
#   Se detiene si el preregistro o algún archivo citado cambió después de la firma, o si falta
#   la enmienda C. Aquí todavía NO se abre el archivo del SRI.
# =====================================================================================
import os, glob, json, time, hashlib, platform
from datetime import datetime
import numpy as np
import pandas as pd
import duckdb

# Todo lo que se imprime con say() queda también en resumen_para_chat.txt (carpeta de la corrida)
SALIDA_TXT, CELDA_ACTUAL = {}, "celda 1"
def say(*args):
    texto = " ".join(str(a) for a in args)
    print(texto)
    SALIDA_TXT.setdefault(CELDA_ACTUAL, []).append(texto)

def guardar_txt():
    if "DIR_RUN" in globals():
        with open(os.path.join(DIR_RUN, "resumen_para_chat.txt"), "w", encoding="utf-8") as f:
            for celda, lineas in SALIDA_TXT.items():
                f.write(f"\n##### {celda.upper()} #####\n" + "\n".join(lineas) + "\n")

def empezar(nombre):
    global CELDA_ACTUAL
    guardar_txt()
    CELDA_ACTUAL = nombre
    SALIDA_TXT[nombre] = []

def sha(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for bloque in iter(lambda: f.read(1 << 20), b""):
            h.update(bloque)
    return h.hexdigest()

# --- rutas ---
SERCOP = r"C:\SERCOP_FINAL"
BASE = os.path.join(SERCOP, "plan_A")
PREREG = os.path.join(SERCOP, "preregistro")
ENMIENDAS = os.path.join(SERCOP, "enmiendas")
CONGELADA = os.path.join(SERCOP, "modelos", "20261016_1017_completo_normal_sup")

# --- parámetros del preregistro y de las enmiendas B y C (no se cambian) ---
SEMILLA = 20261001
B_PERM = 9_999                                   # réplicas de la permutación de H2
TRAMOS = [(1, 1), (2, 3), (4, 7), (8, 15), (16, 31), (32, None)]
NOMBRE_TRAMO = ["1", "2-3", "4-7", "8-15", "16-31", "32+"]
RUC_SIN_MARCA_VIGENTE = ["RUC_OCULTO_14"]        # verificado en el SRI el 9-oct-2026 (no se imprime)
N_LADO, N_RES_PRIO, N_RES_CTRL, N_DOBLE, TOPE = 60, 20, 2, 24, 2
ANIOS_H3 = (2016, 2024)

T_INICIO = time.time()
pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 40)
say(f"Python {platform.python_version()} | numpy {np.__version__} | pandas {pd.__version__} | duckdb {duckdb.__version__}")

# (a) preregistro firmado y sin cambios
firma_path = os.path.join(PREREG, "preregistro_firma.json")
if not os.path.exists(firma_path):
    raise SystemExit("El preregistro no está firmado: no se corre H2 ni H3.")
FIRMA = json.load(open(firma_path, encoding="utf-8"))
if sha(os.path.join(PREREG, FIRMA["documento"])) != FIRMA["huella_documento"]:
    raise SystemExit("El preregistro cambió después de firmarlo: no se corre H2 ni H3.")
for rel, h in FIRMA["huellas_citadas"].items():
    if sha(os.path.join(SERCOP, rel)) != h:
        raise SystemExit(f"{rel} cambió después de la firma: no se corre H2 ni H3.")
say(f"Preregistro verificado: huella {FIRMA['huella_documento'][:16]}, firmado el {FIRMA['fecha_firma']}")

# (b) enmienda C: debe existir, ser única, posterior a la firma y citar este preregistro
cands = sorted(glob.glob(os.path.join(ENMIENDAS, "enmienda_*_c.json")))
if len(cands) != 1:
    raise SystemExit("Falta la enmienda C (o hay más de una). Corre primero la celda de la enmienda C.")
ENM_C_PATH = cands[0]
ENM_C = json.load(open(ENM_C_PATH, encoding="utf-8"))
HUELLA_ENM_C = sha(ENM_C_PATH)
if ENM_C["sigue_a"]["preregistro"] != FIRMA["huella_documento"] or ENM_C["fecha"] < FIRMA["fecha_firma"]:
    raise SystemExit("La enmienda C no corresponde a este preregistro o es anterior a la firma. Avísame.")
if hashlib.sha256(RUC_SIN_MARCA_VIGENTE[0].encode()).hexdigest() != ENM_C["aclaraciones_H2"]["huella_ruc_excluido"]:
    raise SystemExit("El RUC excluido del código no coincide con el registrado en la enmienda C. Avísame.")
say(f"Enmienda C verificada: {os.path.basename(ENM_C_PATH)}, huella {HUELLA_ENM_C[:16]}, del {ENM_C['fecha']}")

# (c) corridas anteriores de este cuaderno (todo es determinista; si hay una, se deja constancia)
PREVIAS = sorted(os.path.basename(d) for d in glob.glob(os.path.join(SERCOP, "modelos", "*_h2_h3"))
                 if glob.glob(os.path.join(d, "registro_h3_muestra.json")) or glob.glob(os.path.join(d, "h2_*")))
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M") + "_h2_h3"
DIR_RUN = os.path.join(SERCOP, "modelos", RUN_ID)
os.makedirs(DIR_RUN, exist_ok=True)
say("Corrida:", RUN_ID)
if PREVIAS:
    say("AVISO: ya hubo corridas de este cuaderno:", ", ".join(PREVIAS),
        "| se deja constancia en el registro. Si fue por un error, dime cuál.")
guardar_txt()

In [ ]:
# =====================================================================================
# CELDA 2. Lista congelada y percentiles dentro de grupo x año (todavía sin el SRI)
#   - percentil a mitad de salto del puntaje dentro de grupo x año: ensamble (H2 principal),
#     índice simple e Isolation Forest (enmienda C)
#   - decil de monto dentro de grupo x año, y percentil del ensamble dentro de grupo x año x decil
# =====================================================================================
empezar("celda 2")
t_celda = time.time()
ruta_p = os.path.join(CONGELADA, "puntajes.parquet").replace("\\", "/")
con = duckdb.connect()
P = con.execute(f"""SELECT ocid, grupo, familia, periodo_calc, anio_proc, monto_final, entidad_ruc,
                          proveedor_ruc_final, ensamble, base, "if" AS iforest, priorizado
                   FROM read_parquet('{ruta_p}')""").df()

def norm_ruc(s):
    t = s.astype("string").str.replace(r"\.0$", "", regex=True).str.replace(r"\D", "", regex=True)
    return t.where(t.str.len() > 0).str.zfill(13)

def pct_mitad_salto(valores, claves):
    """Percentil a mitad de salto dentro de cada grupo de claves (los vacíos quedan vacíos)."""
    r = valores.groupby(claves).rank(method="average")
    n = valores.notna().groupby(claves).transform("sum")
    return (r - 0.5) / n

P["prov"] = norm_ruc(P["proveedor_ruc_final"])
P["ent"] = norm_ruc(P["entidad_ruc"]).fillna("sin_entidad")
P["priorizado"] = P["priorizado"].fillna(False).astype(bool)
P["anio_proc"] = P["anio_proc"].astype(int)
for c in ["monto_final", "ensamble", "base", "iforest"]:
    P[c] = pd.to_numeric(P[c], errors="coerce").astype(float)
GA = [P["grupo"], P["anio_proc"]]
pm = pct_mitad_salto(P["monto_final"], GA)
P["decil_monto"] = (np.minimum(np.floor(pm * 10), 9) + 1).astype("Int64")
P["pct_ens_ga"] = pct_mitad_salto(P["ensamble"], GA)
P["pct_ens_gad"] = pct_mitad_salto(P["ensamble"], GA + [P["decil_monto"]])
P["pct_base_ga"] = pct_mitad_salto(P["base"], GA)
P["pct_if_ga"] = pct_mitad_salto(P["iforest"], GA)

say(f"Lista congelada: {len(P):,} procesos puntuados | priorizados {int(P['priorizado'].sum()):,} | "
    f"años {P['anio_proc'].min()} a {P['anio_proc'].max()}")
say(f"Procesos sin proveedor normalizable: {int(P['prov'].isna().sum()):,}")
say("\nControl de calibración (media del percentil dentro de cada grupo x año con 30 procesos o más; debe dar 0,500):")
filas = []
for c in ["pct_ens_ga", "pct_base_ga", "pct_if_ga", "pct_ens_gad"]:
    claves = GA + [P["decil_monto"]] if c == "pct_ens_gad" else GA
    g_ = P[c].groupby(claves)
    m = g_.mean()[g_.count() >= 30]
    filas.append(dict(percentil=c, celdas=len(m), media_min=m.min(), media_max=m.max()))
say(pd.DataFrame(filas).round(4).to_string(index=False))

POB = P[(P["periodo_calc"] == "pre") & P["pct_ens_ga"].notna() & P["prov"].notna()].copy()
say(f"\nPoblación de H2 (período anterior a la reforma, con puntaje): {len(POB):,} procesos, "
    f"{POB['prov'].nunique():,} proveedores, {POB.groupby(['grupo', 'anio_proc']).ngroups} celdas grupo x año")
say(f"[celda 2: {time.time() - t_celda:.0f} s]")
guardar_txt()

In [ ]:
# =====================================================================================
# CELDA 3. Muestra ciega de H3 (antes de abrir el SRI)
#   60 priorizados + 60 controles del mismo grupo, año y decil de monto, tope de 2 por proveedor
#   y 2 por entidad en cada lado, reservas, códigos neutros, doble revisión y reparto.
#   No se imprime ningún puntaje ni ningún RUC.
# =====================================================================================
empezar("celda 3")
t_celda = time.time()
EL = P[(P["periodo_calc"] == "pre") & P["anio_proc"].between(*ANIOS_H3)].sort_values("ocid")
if EL["decil_monto"].isna().any():
    say(f"AVISO: {int(EL['decil_monto'].isna().sum())} procesos sin monto quedan fuera del sorteo (no tienen decil).")
    EL = EL[EL["decil_monto"].notna()]
PRIO = EL[EL["priorizado"]].reset_index(drop=True)
CTRL = EL[~EL["priorizado"]].reset_index(drop=True)
say(f"Elegibles 2016-2024, período anterior a la reforma: {len(EL):,} | priorizados {len(PRIO):,} | "
    f"no priorizados {len(CTRL):,}")

cuenta = {"priorizado": ({}, {}), "control": ({}, {})}      # lado -> (por proveedor, por entidad)
def cabe(lado, prov, ent):
    cp, ce = cuenta[lado]
    return cp.get(prov, 0) < TOPE and ce.get(ent, 0) < TOPE
def sumar(lado, prov, ent):
    cp, ce = cuenta[lado]
    cp[prov] = cp.get(prov, 0) + 1
    ce[ent] = ce.get(ent, 0) + 1

# (a) priorizados: orden sorteado y aceptación con topes (60 de muestra + 20 de reserva)
orden_p = PRIO.iloc[np.random.default_rng(SEMILLA).permutation(len(PRIO))].reset_index(drop=True)
acept = []
for r in orden_p.itertuples():
    if cabe("priorizado", r.prov, r.ent):
        sumar("priorizado", r.prov, r.ent)
        acept.append(r.Index)
        if len(acept) == N_LADO + N_RES_PRIO:
            break
if len(acept) < N_LADO:
    raise SystemExit(f"Solo {len(acept)} priorizados caben con los topes: avísame.")
if len(acept) < N_LADO + N_RES_PRIO:
    say(f"AVISO: solo hay {len(acept) - N_LADO} priorizados de reserva (se pedían {N_RES_PRIO}).")

# (b) controles: orden sorteado; para cada priorizado, el primero de su grupo, año y decil
orden_c = CTRL.iloc[np.random.default_rng(SEMILLA).permutation(len(CTRL))].reset_index(drop=True)
ESTRATO = orden_c.groupby(["grupo", "anio_proc", "decil_monto"], sort=False).indices
usados = set()
def buscar_control(g, a, d):
    for dist in range(0, 10):
        pos = []
        for dd in {d - dist, d + dist}:
            if 1 <= dd <= 10:
                pos.extend(ESTRATO.get((g, a, dd), []))
        for p in sorted(pos):                                  # orden del sorteo
            if p in usados:
                continue
            r = orden_c.iloc[p]
            if cabe("control", r["prov"], r["ent"]):
                usados.add(p)
                sumar("control", r["prov"], r["ent"])
                return p, dist
    return None, None

filas = []
for k, i in enumerate(acept[:N_LADO], start=1):
    r = orden_p.iloc[i]
    filas.append(dict(ocid=r["ocid"], lado="priorizado", tipo="muestra", par=k, reemplaza_a_par=None,
                      grupo=r["grupo"], familia=r["familia"], anio=int(r["anio_proc"]),
                      decil_objetivo=int(r["decil_monto"]), decil_real=int(r["decil_monto"]), distancia_decil=0,
                      orden_reserva=None))
    p, dist = buscar_control(r["grupo"], int(r["anio_proc"]), int(r["decil_monto"]))
    if p is None:
        raise SystemExit(f"No hay control posible para el par {k}: avísame.")
    c = orden_c.iloc[p]
    filas.append(dict(ocid=c["ocid"], lado="control", tipo="muestra", par=k, reemplaza_a_par=None,
                      grupo=c["grupo"], familia=c["familia"], anio=int(c["anio_proc"]),
                      decil_objetivo=int(r["decil_monto"]), decil_real=int(c["decil_monto"]), distancia_decil=dist,
                      orden_reserva=None))
for j, i in enumerate(acept[N_LADO:], start=1):
    r = orden_p.iloc[i]
    filas.append(dict(ocid=r["ocid"], lado="priorizado", tipo="reserva", par=None, reemplaza_a_par=None,
                      grupo=r["grupo"], familia=r["familia"], anio=int(r["anio_proc"]),
                      decil_objetivo=int(r["decil_monto"]), decil_real=int(r["decil_monto"]), distancia_decil=0,
                      orden_reserva=j))
for k in range(1, N_LADO + 1):
    r = orden_p.iloc[acept[k - 1]]
    for j in range(1, N_RES_CTRL + 1):
        p, dist = buscar_control(r["grupo"], int(r["anio_proc"]), int(r["decil_monto"]))
        if p is None:
            say(f"AVISO: el control del par {k} tiene menos de {N_RES_CTRL} reservas.")
            break
        c = orden_c.iloc[p]
        filas.append(dict(ocid=c["ocid"], lado="control", tipo="reserva", par=None, reemplaza_a_par=k,
                          grupo=c["grupo"], familia=c["familia"], anio=int(c["anio_proc"]),
                          decil_objetivo=int(r["decil_monto"]), decil_real=int(c["decil_monto"]),
                          distancia_decil=dist, orden_reserva=j))
M = pd.DataFrame(filas)

# (c) códigos neutros sorteados: R-001 a R-120 para la muestra; los siguientes para las reservas
rng_m = np.random.default_rng(SEMILLA)
mu = M.index[M["tipo"] == "muestra"].to_numpy()
re_ = M.index[M["tipo"] == "reserva"].to_numpy()
M["codigo"] = ""
M.loc[rng_m.permutation(mu), "codigo"] = [f"R-{n:03d}" for n in range(1, len(mu) + 1)]
M.loc[rng_m.permutation(re_), "codigo"] = [f"R-{n:03d}" for n in range(len(mu) + 1, len(mu) + len(re_) + 1)]
cod_par = M[(M["tipo"] == "muestra") & (M["lado"] == "control")].set_index("par")["codigo"]
M["reemplaza_a"] = M["reemplaza_a_par"].map(cod_par)

# (d) doble revisión (24 sorteados) y reparto del resto, 48 y 48
rng_d = np.random.default_rng(SEMILLA)
cods = np.sort(M.loc[mu, "codigo"].to_numpy())
doble = set(rng_d.choice(cods, size=N_DOBLE, replace=False))
resto = rng_d.permutation([c for c in cods if c not in doble])
mitad = len(resto) // 2
M["doble_revision"] = M["codigo"].isin(doble)
M["revisor"] = ""
M.loc[M["codigo"].isin(doble), "revisor"] = "A y B"
M.loc[M["codigo"].isin(resto[:mitad]), "revisor"] = "A"
M.loc[M["codigo"].isin(resto[mitad:]), "revisor"] = "B"
M["codigo_proceso"] = M["ocid"].str.replace(r"^ocds-[^-]+-", "", regex=True).str.replace(r"-\d+$", "", regex=True)

# (e) archivos: dos hojas para revisores (solo datos neutros), guía, clave sellada y registro
DIR_REV = os.path.join(DIR_RUN, "revision_h3")
DIR_CLAVE = os.path.join(DIR_RUN, "CLAVE_H3_NO_COMPARTIR")
os.makedirs(DIR_REV, exist_ok=True)
os.makedirs(DIR_CLAVE, exist_ok=True)
ITEMS = ["A1_falta_documento_esencial", "A2_plazo_menor_al_minimo", "A3_no_es_menor_precio",
         "A4_descalifica_sin_motivo", "A5_actividad_no_relacionada", "A6_fraccionamiento_30_dias",
         "A7_contrato_mayor_al_adjudicado"]
EXTRA = ["evidencia_enlaces_o_capturas", "items_tipo_B", "minutos", "observaciones"]

def guardar_hoja(tabla, path):
    """Excel con lista desplegable Sí / No / No verificable; si no hay openpyxl, CSV con ';'."""
    try:
        import openpyxl
        from openpyxl.utils import get_column_letter
        from openpyxl.worksheet.datavalidation import DataValidation
        tabla.to_excel(path, index=False, sheet_name="revision")
        wb = openpyxl.load_workbook(path)
        ws = wb["revision"]
        dv = DataValidation(type="list", formula1='"Sí,No,No verificable"', allow_blank=True)
        ws.add_data_validation(dv)
        for j, col in enumerate(tabla.columns, start=1):
            letra = get_column_letter(j)
            ws.column_dimensions[letra].width = 14 if col in ITEMS else max(12, min(45, len(col) + 4))
            if col in ITEMS:
                dv.add(f"{letra}2:{letra}{len(tabla) + 1}")
        ws.freeze_panes = "C2"
        wb.save(path)
        return path
    except ImportError:
        destino = path.replace(".xlsx", ".csv")
        tabla.to_csv(destino, index=False, sep=";", encoding="utf-8-sig")
        return destino

HOJAS = {}
for rev in ["A", "B"]:
    t = M[(M["tipo"] == "muestra") & M["revisor"].str.contains(rev)].sort_values("codigo")
    t = t[["codigo", "ocid", "codigo_proceso", "familia", "anio"]].copy()
    for c in ITEMS + EXTRA:
        t[c] = ""
    HOJAS[rev] = guardar_hoja(t, os.path.join(DIR_REV, f"para_revisor_{rev}.xlsx"))

texto_prereg = open(os.path.join(PREREG, FIRMA["documento"]), encoding="utf-8").read()
ini = texto_prereg.find("## Anexo B")
GUIA = os.path.join(DIR_REV, "guia_revision_anexo_B.txt")
with open(GUIA, "w", encoding="utf-8") as f:
    f.write(texto_prereg[ini:] if ini >= 0 else "No se encontró el Anexo B en el preregistro.")
    f.write("\n\nCómo llenar la hoja: en cada ítem A1 a A7 elige Sí / No / No verificable. Un 'Sí' necesita el\n"
            "enlace o la captura en la columna de evidencia. Si la ficha no abre o no tiene documentos, marca\n"
            "'No verificable' en todos los ítems y avisa a quien preparó la muestra: te dará un código de reserva.\n")

CLAVE = os.path.join(DIR_CLAVE, "clave_h3.csv")
M[["codigo", "ocid", "codigo_proceso", "lado", "tipo", "par", "reemplaza_a", "orden_reserva", "grupo", "familia",
   "anio", "decil_objetivo", "decil_real", "distancia_decil", "doble_revision", "revisor"]] \
    .sort_values("codigo").to_csv(CLAVE, index=False, encoding="utf-8-sig")
with open(os.path.join(DIR_CLAVE, "LEEME.txt"), "w", encoding="utf-8") as f:
    f.write("Esta carpeta dice qué código es priorizado y cuál es control. No se comparte con los revisores\n"
            "hasta que entreguen sus hojas. Si un proceso no es verificable: si es priorizado, se reemplaza por\n"
            "el siguiente priorizado de reserva (orden_reserva 1, 2, ...); si es control, por su reserva\n"
            "(columna reemplaza_a = su código, orden_reserva 1 y luego 2). Al revisor se le da solo el código\n"
            "y el ocid de la reserva, y se anota el cambio.\n")

REG_H3 = dict(corrida=RUN_ID, fecha=datetime.now().isoformat(timespec="seconds"), semilla=SEMILLA,
              preregistro=FIRMA["huella_documento"], enmienda_c=HUELLA_ENM_C,
              elegibles=int(len(EL)), priorizados_elegibles=int(len(PRIO)),
              muestra=dict(priorizados=int(((M["tipo"] == "muestra") & (M["lado"] == "priorizado")).sum()),
                           controles=int(((M["tipo"] == "muestra") & (M["lado"] == "control")).sum())),
              reservas=dict(priorizados=int(((M["tipo"] == "reserva") & (M["lado"] == "priorizado")).sum()),
                            controles=int(((M["tipo"] == "reserva") & (M["lado"] == "control")).sum())),
              huellas={"clave_h3.csv": sha(CLAVE), "guia": sha(GUIA),
                       **{os.path.basename(v): sha(v) for v in HOJAS.values()}})
with open(os.path.join(DIR_RUN, "registro_h3_muestra.json"), "w", encoding="utf-8") as f:
    json.dump(REG_H3, f, ensure_ascii=False, indent=1)

# (f) controles del sorteo (sin puntajes ni RUC)
mm = M[M["tipo"] == "muestra"]
say(f"Muestra: {REG_H3['muestra']} | reservas: {REG_H3['reservas']}")
say("Controles con decil vecino:", int((mm.loc[mm['lado'] == 'control', 'distancia_decil'] > 0).sum()),
    "| distancias:", mm.loc[mm["lado"] == "control", "distancia_decil"].value_counts().sort_index().to_dict())
for lado in ["priorizado", "control"]:
    s = M[M["lado"] == lado].merge(P[["ocid", "prov", "ent"]], on="ocid")
    say(f"  {lado:<10}: máximo por proveedor {s['prov'].value_counts().max()}, por entidad "
        f"{s['ent'].value_counts().max()} (tope {TOPE}, muestra y reservas juntas)")
comp = mm.pivot_table(index="grupo", columns="lado", values="ocid", aggfunc="count", fill_value=0)
say("\nComposición por grupo (muestra):")
say(comp.to_string())
say("\nComposición por año (muestra):")
say(mm.pivot_table(index="anio", columns="lado", values="ocid", aggfunc="count", fill_value=0).T.to_string())
mo = mm.merge(P[["ocid", "monto_final"]], on="ocid").groupby("lado")["monto_final"].median()
say("\nMonto mediano por lado (USD):", {k: round(v, 2) for k, v in mo.items()})
say(f"\nRevisor A: {int(M['revisor'].str.contains('A').sum())} procesos | revisor B: "
    f"{int(M['revisor'].str.contains('B').sum())} | con doble revisión: {len(doble)}")
say("Hojas para revisores:", ", ".join(os.path.basename(v) for v in HOJAS.values()), "en", DIR_REV)
say(f"Clave sellada: {CLAVE}\n  huella {REG_H3['huellas']['clave_h3.csv']}")
say("No abras la clave delante de los revisores ni se la envíes hasta que entreguen sus hojas.")
say(f"[celda 3: {time.time() - t_celda:.0f} s]")
guardar_txt()

In [ ]:
# =====================================================================================
# CELDA 4. Se abre el archivo del SRI (única vez) y se registran los valores críticos de H2
#   ANTES de calcular T. Aquí solo se usan cuántos procesos tiene cada contribuyente, nunca
#   sus puntajes. No se imprime ningún RUC.
# =====================================================================================
empezar("celda 4")
t_celda = time.time()
SRI_PATH = os.path.join(BASE, "planA_validacion_sri.parquet")
if not os.path.exists(SRI_PATH):
    raise SystemExit(f"No encuentro {SRI_PATH}.")
HUELLA_SRI = sha(SRI_PATH)
SRI = con.execute(f"SELECT * FROM read_parquet('{SRI_PATH.replace(chr(92), '/')}')").df()
say(f"planA_validacion_sri abierto: {len(SRI)} filas | huella {HUELLA_SRI[:16]}")
say("Columnas:", ", ".join(SRI.columns))

col_ruc = None
for c in sorted([c for c in SRI.columns if "ruc" in c.lower() and "entidad" not in c.lower()],
                key=lambda c: (c.lower() != "ruc", c)):
    v = norm_ruc(SRI[c]).dropna()
    if len(v) == len(SRI) and v.str.len().eq(13).all() and v.nunique() == 15:
        col_ruc = c
        break
if col_ruc is None:
    raise SystemExit("No encuentro una columna con los 15 RUC. Pégame solo la línea 'Columnas' de arriba.")
RUC15 = sorted(set(norm_ruc(SRI[col_ruc]).dropna()))
if not set(RUC_SIN_MARCA_VIGENTE) <= set(RUC15):
    raise SystemExit("El RUC sin marca vigente no está en el archivo: no sigo. Avísame.")
RUC14 = [r for r in RUC15 if r not in RUC_SIN_MARCA_VIGENTE]
say(f"Columna de RUC: '{col_ruc}' | contribuyentes en el archivo: {len(RUC15)} | con marca vigente: {len(RUC14)}")

# Dónde están los procesos de los 15 (solo conteos): base completa, puntuados y población de H2
ruta_pm = os.path.join(BASE, "planA_procesos_modelo.parquet").replace("\\", "/")
PMk = con.execute(f"SELECT ocid, proveedor_ruc_final, fecha_inicio_tender FROM read_parquet('{ruta_pm}')").df()
PMk["prov"] = norm_ruc(PMk["proveedor_ruc_final"])
PMk["anio"] = pd.to_datetime(PMk["fecha_inicio_tender"], errors="coerce").dt.year
cod_c = {r: f"C{j:02d}" for j, r in enumerate(RUC15, start=1)}      # código anónimo de cada contribuyente
n_pob = POB.groupby("prov").size()

def tramo(n):
    for i, (lo, hi) in enumerate(TRAMOS):
        if n >= lo and (hi is None or n <= hi):
            return i
    return None

filas = []
for r in RUC15:
    base_r = PMk[PMk["prov"] == r]
    filas.append(dict(contribuyente=cod_c[r], marca_vigente=r not in RUC_SIN_MARCA_VIGENTE,
                      en_la_base=len(base_r), de_2015=int((base_r["anio"] < 2016).sum()),
                      puntuados=int((P["prov"] == r).sum()), poblacion_H2=int(n_pob.get(r, 0)),
                      tramo=NOMBRE_TRAMO[tramo(n_pob[r])] if r in n_pob.index else "-"))
TAB_C = pd.DataFrame(filas)
say("\nProcesos de cada contribuyente (código anónimo; solo conteos, sin puntajes):")
say(TAB_C.to_string(index=False))
say(f"Totales: en la base {TAB_C['en_la_base'].sum()} | de 2015 {TAB_C['de_2015'].sum()} | puntuados "
    f"{TAB_C['puntuados'].sum()} | población H2 {TAB_C['poblacion_H2'].sum()} "
    f"(14 con marca vigente: {TAB_C.loc[TAB_C['marca_vigente'], 'poblacion_H2'].sum()})")
del PMk

# Donantes por tramo: proveedores de la población que no están entre los 15 del archivo
DON = n_pob[~n_pob.index.isin(RUC15)]
POOL = {i: np.sort(DON.index[[tramo(n) == i for n in DON.to_numpy()]].to_numpy()) for i in range(len(TRAMOS))}
say("\nDonantes por tramo:", {NOMBRE_TRAMO[i]: len(v) for i, v in POOL.items()})

def sortear(rucs):
    """Devuelve, para cada réplica, la lista de proveedores donantes (sin reposición dentro de cada tramo)."""
    rng = np.random.default_rng(SEMILLA)
    por_tramo = pd.Series([tramo(n_pob[r]) for r in rucs if r in n_pob.index]).value_counts().sort_index()
    bloques = []
    for i, m in por_tramo.items():
        pool = POOL[i]
        if len(pool) < m:
            raise SystemExit(f"El tramo {NOMBRE_TRAMO[i]} tiene menos donantes que contribuyentes.")
        d = rng.integers(0, len(pool), size=(B_PERM, m))
        while True:
            rep = np.array([len(np.unique(f)) < m for f in d])
            if not rep.any():
                break
            d[rep] = rng.integers(0, len(pool), size=(int(rep.sum()), m))
        bloques.append(pool[d])
    return np.hstack(bloques), por_tramo

def nula(columna, donantes):
    """Distribución de T bajo la hipótesis nula: mediana de los percentiles de los donantes."""
    sub = POB[POB[columna].notna()].sort_values("prov")
    prov = sub["prov"].to_numpy()
    val = sub[columna].to_numpy(dtype=float)
    ini_, fin_ = np.searchsorted(prov, donantes, side="left"), np.searchsorted(prov, donantes, side="right")
    T0 = np.empty(B_PERM)
    for b in range(B_PERM):
        T0[b] = np.median(np.concatenate([val[i:j] for i, j in zip(ini_[b], fin_[b])]))
    return T0

DON14, TR14 = sortear(RUC14)
DON15, TR15 = sortear(RUC15)
VARIANTES = [  # nombre, columna de percentil, contribuyentes, estatus
    ("H2 principal: ensamble, grupo x año, 14", "pct_ens_ga", "14", "confirmatoria"),
    ("Sensibilidad: ensamble, grupo x año x decil de monto, 14", "pct_ens_gad", "14", "sensibilidad"),
    ("Sensibilidad: ensamble, grupo x año, 15", "pct_ens_ga", "15", "sensibilidad"),
    ("Enmienda C: índice simple, grupo x año, 14", "pct_base_ga", "14", "exploratoria"),
    ("Enmienda C: Isolation Forest, grupo x año, 14", "pct_if_ga", "14", "exploratoria"),
]
NULAS, filas = {}, []
for nombre, col, quien, estatus in VARIANTES:
    T0 = nula(col, DON14 if quien == "14" else DON15)
    NULAS[nombre] = T0
    filas.append(dict(variante=nombre, estatus=estatus, critico_95=float(np.quantile(T0, 0.95)),
                      critico_975=float(np.quantile(T0, 0.975)), media_nula=float(T0.mean())))
VC = pd.DataFrame(filas)
NUL_PATH = os.path.join(DIR_RUN, "h2_distribuciones_nulas.csv")
pd.DataFrame(NULAS).to_csv(NUL_PATH, index=False)
VC_PATH = os.path.join(DIR_RUN, "h2_valores_criticos.json")
with open(VC_PATH, "w", encoding="utf-8") as f:
    json.dump(dict(fecha=datetime.now().isoformat(timespec="seconds"), corrida=RUN_ID, semilla=SEMILLA,
                   replicas=B_PERM, preregistro=FIRMA["huella_documento"], enmienda_c=HUELLA_ENM_C,
                   huella_sri=HUELLA_SRI, huella_nulas=sha(NUL_PATH),
                   tramos_14={NOMBRE_TRAMO[i]: int(m) for i, m in TR14.items()},
                   tramos_15={NOMBRE_TRAMO[i]: int(m) for i, m in TR15.items()},
                   valores_criticos=VC.to_dict(orient="records")), f, ensure_ascii=False, indent=1)
HUELLA_VC = sha(VC_PATH)
say("\nContribuyentes por tramo (14):", {NOMBRE_TRAMO[i]: int(m) for i, m in TR14.items()})
say("\nValores críticos registrados ANTES de calcular T (percentil 95 y 97,5 de la distribución nula):")
say(VC.round(4).to_string(index=False))
say(f"\nRegistrado en {VC_PATH}\n  huella {HUELLA_VC}\n  a las {datetime.now():%H:%M:%S}")
say("Antes de correr la celda 5, mándate esta huella por correo (o al grupo).")
say(f"[celda 4: {time.time() - t_celda:.0f} s]")
guardar_txt()

In [ ]:
# =====================================================================================
# CELDA 5. Prueba de H2: T de los contribuyentes, valor p y lectura
#   p unilateral = (1 + réplicas con T* >= T) / 10.000. Holm con H3 se aplica cuando esté H3.
# =====================================================================================
empezar("celda 5")
if sha(VC_PATH) != HUELLA_VC:
    raise SystemExit("El archivo de valores críticos cambió después de registrarlo: no sigo.")
filas = []
for nombre, col, quien, estatus in VARIANTES:
    rucs = RUC14 if quien == "14" else RUC15
    s = POB.loc[POB["prov"].isin(rucs) & POB[col].notna(), col]
    T = float(np.median(s)) if len(s) else np.nan
    T0 = NULAS[nombre]
    p = (1 + int((T0 >= T).sum())) / (B_PERM + 1) if len(s) else np.nan
    v = VC.set_index("variante").loc[nombre]
    filas.append(dict(variante=nombre, estatus=estatus, procesos=len(s),
                      contribuyentes=int(POB.loc[POB["prov"].isin(rucs), "prov"].nunique()),
                      T=T, critico_95=v["critico_95"], p_unilateral=p))
RES_H2 = pd.DataFrame(filas)
say("Resultados de H2 (T = percentil mediano de los procesos de los contribuyentes; 0,5 = como el resto):")
say(RES_H2.round(4).to_string(index=False))

pr = RES_H2.iloc[0]
say("\n" + "=" * 100)
say(f"H2 PRINCIPAL: T = {pr['T']:.4f} con {pr['procesos']} procesos de {pr['contribuyentes']} contribuyentes | "
    f"valor crítico {pr['critico_95']:.4f} | p unilateral = {pr['p_unilateral']:.4f}")
if pr["p_unilateral"] <= 0.025:
    lect = "p <= 0,025: H2 se rechaza con Holm sea cual sea el p de H3."
elif pr["p_unilateral"] <= 0.05:
    lect = "0,025 < p <= 0,05: H2 se rechaza solo si H3 sale con p <= 0,025 (Holm)."
else:
    lect = ("p > 0,05: H2 no se rechaza. Según el preregistro se informa como no detectable con los datos "
            "disponibles, no como ausencia de efecto.")
say("Lectura con Holm:", lect)
say("Recuerda: como H1 no se cumplió, la hipótesis general no queda respaldada, salga lo que salga en H2 y H3.")
say("Las variantes 'exploratoria' (enmienda C) no entran a Holm ni a la decisión general.")

# descriptivo por contribuyente, con código anónimo (sin RUC)
desc = (POB[POB["prov"].isin(RUC15)].assign(contribuyente=lambda t: t["prov"].map(cod_c))
        .groupby("contribuyente").agg(procesos=("ocid", "size"), mediana_ensamble=("pct_ens_ga", "median"),
                                      mediana_indice=("pct_base_ga", "median"), mediana_if=("pct_if_ga", "median"),
                                      en_la_lista=("priorizado", "sum")))
say("\nDescriptivo por contribuyente (percentil dentro de grupo x año; código anónimo):")
say(desc.round(3).to_string())
RES_H2.to_csv(os.path.join(DIR_RUN, "h2_resultados.csv"), index=False)
desc.to_csv(os.path.join(DIR_RUN, "h2_por_contribuyente.csv"))
REGISTRO = dict(corrida=RUN_ID, fecha=datetime.now().isoformat(timespec="seconds"), corridas_previas=PREVIAS,
                preregistro=FIRMA["huella_documento"], enmienda_c=HUELLA_ENM_C, huella_sri=HUELLA_SRI,
                huella_valores_criticos=HUELLA_VC, h3_muestra=REG_H3["huellas"],
                h2=RES_H2.to_dict(orient="records"), lectura_holm=lect,
                versiones=dict(python=platform.python_version(), numpy=np.__version__, pandas=pd.__version__,
                               duckdb=duckdb.__version__),
                minutos=round((time.time() - T_INICIO) / 60, 1))
with open(os.path.join(DIR_RUN, "registro_h2.json"), "w", encoding="utf-8") as f:
    json.dump(REGISTRO, f, ensure_ascii=False, indent=1, default=float)
say(f"\nArchivos en {DIR_RUN} | {REGISTRO['minutos']} min")
guardar_txt()
say("Adjunta en el chat el archivo resumen_para_chat.txt de esa carpeta (no la clave de H3).")
guardar_txt()